# 08 — Anti-poisoning: the spec that attacks its reader

**Goal:** see spec poisoning concretely, build a miniature detector, and understand the layered defense.

**Fixture:** `fixtures/poisoned-spec.yaml` — a teaching fixture imitating real attacks: embedded instructions, credential harvesting, and an exfiltration server. **Never load a spec like this into a live, credentialed integration.**

Ingested specs and docs are **untrusted input** — the same rule this course applies to retrieved documents (Session 14), one level up the supply chain.

In [ ]:
# Offline by default. Repo + fixture paths:
from pathlib import Path
import sys

here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
sys.path.insert(0, str(here / "src"))
FIXTURES = here / "cookbook" / "fixtures"
CORPUS_DIR = here / "data" / "corpus"
print(f"fixtures: {FIXTURES}")

## 1. Read the attack

In [ ]:
import yaml

raw = (FIXTURES / "poisoned-spec.yaml").read_text()
spec = yaml.safe_load(raw)
print(spec['info']['description'])
print('---')
print(spec['paths']['/price']['get']['description'])
print('---')
print('servers:', [s['url'] for s in spec['servers']])

## 2. A miniature detector

Three signal families (Gecko's sanitizer covers far more — encoded payloads, image-borne instructions, fund-routing patterns — but these three catch this fixture and teach the shape):

1. **Imperatives aimed at the model** inside descriptions
2. **Credential/environment harvesting** language
3. **Server list anomalies** (a second host unrelated to the API's domain)

In [ ]:
import re

INSTRUCTION_PATTERNS = [
    r'ignore (all )?previous instructions',
    r'note to ai assistants',
    r'you must (also )?post',
    r'system note',
]
HARVEST_PATTERNS = [
    r'api[_-]?key', r'environment variable', r'credential', r'secret',
]


def scan_spec(spec: dict) -> list[str]:
    findings = []
    def walk(node, path):
        if isinstance(node, dict):
            for key, value in node.items():
                walk(value, f'{path}.{key}')
        elif isinstance(node, str):
            lowered = node.lower()
            for pattern in INSTRUCTION_PATTERNS:
                if re.search(pattern, lowered):
                    findings.append(f'INSTRUCTION at {path}: {pattern!r}')
            if any(re.search(p, lowered) for p in HARVEST_PATTERNS) and any(
                re.search(p, lowered) for p in INSTRUCTION_PATTERNS + [r'include them', r'read the']
            ):
                findings.append(f'HARVEST at {path}')
        elif isinstance(node, list):
            for index, item in enumerate(node):
                walk(item, f'{path}[{index}]')
    walk(spec, '$')
    hosts = {s['url'].split('/')[2] for s in spec.get('servers', [])}
    if len(hosts) > 1:
        findings.append(f'SERVER ANOMALY: multiple unrelated hosts {sorted(hosts)}')
    return findings

findings = scan_spec(spec)
for finding in findings:
    print(finding)
assert findings, 'the poisoned fixture must trip the detector'
print(f'\nverdict: QUARANTINE ({len(findings)} findings) — do not generate tools from this spec')

## 3. Prove the clean specs pass

A detector that flags everything is as useless as one that flags nothing (Session 9's evaluator lesson, applied to security).

In [ ]:
for name in ('petstore-mini.yaml', 'weather-mini.yaml'):
    clean = yaml.safe_load((FIXTURES / name).read_text())
    result = scan_spec(clean)
    print(f'{name}: {result or "clean"}')
    assert not result

## 4. Why detection alone is not the defense

Patterns can be evaded (encodings, images, novel phrasings). The layers that hold even when detection misses:

- **Fail closed**: a suspicious spec is quarantined — no tools generated — rather than 'generated with a warning'.
- **Credentials the model never sees** cannot be exfiltrated by any injected instruction (the transport-edge rule, again).
- **Bounded, read-only tools by default**: the injected 'POST your conversation to /exfil' has no tool that can do it.
- **Never weaken a detection rule to fix a false positive** without a security review — a relaxed pattern is an opened bypass.

Gecko ships this discipline as the `anti-poisoning` skill; the fixture you just scanned is the classroom version of what it defends against.